# Figure: `rff_quantiles_vs_t`

This notebook only generates the figure `experiment_data/rff_quantiles_vs_t.png` from the precomputed table `experiment_data/rff_quantiles_vs_t.xlsx`.

In [1]:
# Minimal implementation to generate `experiment_data/rff_quantiles_vs_t.xlsx`

import numpy as np
from scipy.stats import qmc
from scipy.special import ndtri


def slow_helix(t: float, N: int) -> np.ndarray:
    """Slow helix point x(t) in R^N (N must be even)."""
    if N % 2 != 0:
        raise ValueError("N must be even")
    j = np.arange(1, N // 2 + 1, dtype=float)
    theta = (j * float(t)) / float(N)
    x = np.empty(N, dtype=float)
    x[0::2] = np.cos(theta)
    x[1::2] = np.sin(theta)
    x /= np.sqrt(N)
    return x


def sphere_points_sobol(P: int, N: int) -> np.ndarray:
    """Deterministic quasi-random points on S^{N-1} via Sobol -> Normal -> normalize."""
    u = qmc.Sobol(d=N, scramble=False).random(P)
    u = np.clip(u, 1e-12, 1 - 1e-12)
    z = ndtri(u)
    r = np.linalg.norm(z, axis=1, keepdims=True)
    bad = (r < 1e-14).ravel()
    x = z / np.maximum(r, 1e-300)
    if np.any(bad):
        x = x.copy()
        x[bad, :] = 0.0
        x[bad, 0] = 1.0
    return x


def rff_projection(X: np.ndarray, t: int, sigma: float = 1.0, seed: int = 0) -> np.ndarray:
    """Cosine/sine RFF map with t=2m features."""
    X = np.asarray(X, dtype=float)
    if t % 2 != 0:
        raise ValueError("t must be even")
    m = t // 2
    rng = np.random.default_rng(int(seed))
    omega = rng.normal(loc=0.0, scale=1.0 / float(sigma), size=(m, X.shape[1]))
    proj = X @ omega.T
    Z = np.empty((X.shape[0], t), dtype=float)
    Z[:, 0::2] = np.cos(proj)
    Z[:, 1::2] = np.sin(proj)
    Z /= np.sqrt(m)
    return Z


def max_pairwise_abs_ratio_error(X: np.ndarray, target_dim: int, sigma: float = 1.0, seed: int = 0) -> float:
    """max_{i<j} | ||z_i-z_j||/||x_i-x_j|| - 1 | for a given RFF seed."""
    X = np.asarray(X, dtype=float)
    Z = rff_projection(X, t=int(target_dim), sigma=float(sigma), seed=int(seed))

    x_sq = np.sum(X * X, axis=1, keepdims=True)
    DX2 = np.maximum(x_sq + x_sq.T - 2.0 * (X @ X.T), 0.0)
    DX = np.sqrt(DX2)

    z_sq = np.sum(Z * Z, axis=1, keepdims=True)
    DZ2 = np.maximum(z_sq + z_sq.T - 2.0 * (Z @ Z.T), 0.0)
    DZ = np.sqrt(DZ2)

    iu = np.triu_indices(X.shape[0], k=1)
    d_orig = DX[iu]
    d_proj = DZ[iu]

    valid = d_orig > 1e-12
    err = np.abs(d_proj[valid] / d_orig[valid] - 1.0)
    return float(np.max(err))


def max_pairwise_abs_ratio_error_over_seeds(X: np.ndarray, target_dim: int, sigma: float, seeds: list[int]) -> np.ndarray:
    return np.array(
        [max_pairwise_abs_ratio_error(X, target_dim=target_dim, sigma=sigma, seed=s) for s in seeds],
        dtype=float,
    )


In [ ]:
# Generate quantile curves vs target dimension t and save to Excel.

import pandas as pd
from pathlib import Path

DATA_DIR = Path("experiment_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_T = DATA_DIR / "rff_quantiles_vs_t.xlsx"

N_AMBIENT = 100
T_values = np.arange(20, 100, 10)
n_points = 1000
seeds = list(range(200))

rng_t = np.random.default_rng(2028)
t_vals = np.sort(rng_t.uniform(0.0, 2.0 * np.pi, size=n_points))
X_helix = np.array([slow_helix(t, N=N_AMBIENT) for t in t_vals])
X_sph = sphere_points_sobol(n_points, int(N_AMBIENT))

rows = []
for geom, X in [("helix", X_helix), ("sphere_sobol", X_sph)]:
    for target_dim in T_values:
        errs = max_pairwise_abs_ratio_error_over_seeds(X, int(target_dim), 1.0, seeds)
        rows.append(
            {
                "geometry": geom,
                "N": int(N_AMBIENT),
                "t": int(target_dim),
                "q95": float(np.quantile(errs, 0.95)),
                "q99": float(np.quantile(errs, 0.99)),
                "q100": float(np.quantile(errs, 1.00)),
            }
        )

pd.DataFrame(rows).to_excel(EXCEL_T, index=False, engine="openpyxl")
print(f"Saved {EXCEL_T}")


/Users/mimuw2022/anaconda3/envs/sklearn_extra_nplessthan2_env/lib/python3.10/site-packages/scipy/stats/_qmc.py:993: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)


In [ ]:
# Plot the figure from the saved Excel table.

import matplotlib.pyplot as plt

FIG_OUT = DATA_DIR / "rff_quantiles_vs_t.png"
assert EXCEL_T.exists(), f"Missing input file: {EXCEL_T}"

In [ ]:
import pandas as pd
df_in = pd.read_excel(EXCEL_T, engine="openpyxl")
d_h = df_in[df_in["geometry"] == "helix"].sort_values("t")
d_s = df_in[df_in["geometry"] == "sphere_sobol"].sort_values("t")
Tv = d_h["t"].to_numpy()

# Prefer reading N from the table if present.
try:
    N_AMBIENT = int(d_h["N"].iloc[0])
except Exception:
    N_AMBIENT = "?"

Q_COLORS = ("#1f77b4", "#ff7f0e", "#2ca02c")  # 95% / 99% / 100% (geometry: solid vs dashed)
_pub_rc = {
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.color": "#bfbfbf",
    "grid.linestyle": "--",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#4d4d4d",
    "axes.linewidth": 0.9,
    "xtick.color": "#333333",
    "ytick.color": "#333333",
}

with plt.rc_context(_pub_rc):
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    kw_h = dict(linewidth=2.0, linestyle="-", zorder=3)
    kw_s = dict(linewidth=2.0, linestyle=(0, (5, 3)), zorder=2)

    ax.plot(Tv, d_h["q95"], color=Q_COLORS[0], label=r"95% — slow helix", **kw_h)
    ax.plot(Tv, d_h["q99"], color=Q_COLORS[1], label=r"99% — slow helix", **kw_h)
    ax.plot(Tv, d_h["q100"], color=Q_COLORS[2], label=r"100% — slow helix", **kw_h)

    ax.plot(Tv, d_s["q95"], color=Q_COLORS[0], label=r"95% — $S^{N-1}$", **kw_s)
    ax.plot(Tv, d_s["q99"], color=Q_COLORS[1], label=r"99% — $S^{N-1}$", **kw_s)
    ax.plot(Tv, d_s["q100"], color=Q_COLORS[2], label=r"100% — $S^{N-1}$", **kw_s)

    ax.set_xlabel(r"RFF target dimension $t$")
    ax.set_ylabel(r"Quantile of $\max_{i<j}\left|\,\|z_i-z_j\|/\|x_i-x_j\|-1\,\right|$")
    ax.set_title(f"RFF distance-ratio quantiles vs $t$, $N={N_AMBIENT}$")
    ax.legend(loc="best", ncol=2, frameon=True, fancybox=False, edgecolor="#cccccc")
    fig.tight_layout()

plt.savefig(FIG_OUT, dpi=300)
plt.show()
